<a href="https://colab.research.google.com/github/JouichatKhadija/.github.io/blob/main/TP1_deep_learning_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Deep Learning — TP Pratique avec PyTorch

## Séance 1 : Des Tenseurs au XOR

**Objectifs :**
- Manipuler les tenseurs PyTorch
- Comprendre le forward pass d'un neurone
- Visualiser les fonctions d'activation
- Implémenter la descente de gradient à la main puis avec PyTorch
- Résoudre le problème XOR

---

In [ ]:
# Installation (décommenter si nécessaire)
# !pip install torch matplotlib numpy

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

# Style des graphiques
plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA disponible : {torch.cuda.is_available()}")

PyTorch version : 2.10.0+cpu
CUDA disponible : False


---
# 📦 Partie 1 : Les Tenseurs

Un tenseur est un conteneur de nombres avec des dimensions.

| Type | Dimension | Exemple |
|------|-----------|----------|
| Scalaire | 0 | Température = 42 |
| Vecteur | 1 | Pixel RGB = [255, 128, 0] |
| Matrice | 2 | Image N&B 28×28 |
| Tenseur 3D+ | 3+ | Image couleur 224×224×3 |

## 1.1 Création de tenseurs

In [ ]:
# ── Scalaire (dimension 0) ──
scalaire = torch.tensor(42.0)
print(f"Scalaire : {scalaire}")
print(f"  Dimension : {scalaire.dim()}")
print(f"  Shape     : {scalaire.shape}")
print()

Scalaire : 42.0
  Dimension : 0
  Shape     : torch.Size([])



In [ ]:
# ── Vecteur (dimension 1) ──
vecteur = torch.tensor([255, 128, 0])  # Un pixel RGB
print(f"Vecteur : {vecteur}")
print(f"  Dimension : {vecteur.dim()}")
print(f"  Shape     : {vecteur.shape}")
print(f"  Taille    : {vecteur.shape[0]} éléments")
print()

In [ ]:
# ── Matrice (dimension 2) ──
matrice = torch.tensor([[1, 2, 3],
                        [4, 5, 6]])
print(f"Matrice :\n{matrice}")
print(f"  Dimension : {matrice.dim()}")
print(f"  Shape     : {matrice.shape}  (2 lignes × 3 colonnes)")
print()

In [ ]:
# ── Tenseur 3D (comme une image couleur) ──
image = torch.rand(224, 224, 3)  # Image 224×224 RGB (valeurs aléatoires)
print(f"Image RGB :")
print(f"  Dimension : {image.dim()}")
print(f"  Shape     : {image.shape}")
print(f"  Nb valeurs: {image.numel():,}  (= 224 × 224 × 3)")
print()

# ── Batch de 32 images (tenseur 4D) ──
batch = torch.rand(32, 224, 224, 3)
print(f"Batch de 32 images :")
print(f"  Dimension : {batch.dim()}")
print(f"  Shape     : {batch.shape}")
print(f"  Nb valeurs: {batch.numel():,}  (= 32 × 224 × 224 × 3)")

NameError: name 'torch' is not defined

### ✏️ Exercice 1.1
Créez un tenseur représentant une phrase de 15 mots, chaque mot encodé en 768 dimensions (comme BERT).
Affichez sa shape et le nombre total de valeurs.

In [ ]:
# ══ VOTRE CODE ICI ══
phrase = ...  # Indice : torch.rand(?, ?)



<details><summary>💡 Solution</summary>

```python
phrase = torch.rand(15, 768)
print(f"Shape : {phrase.shape}")
print(f"Nb valeurs : {phrase.numel():,}")  # 11,520
```
</details>

## 1.2 Opérations sur les tenseurs

In [ ]:
# ══════════════════════════════════════
# OPÉRATION 1 : Addition élément par élément
# ══════════════════════════════════════
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])
c = a + b

print("Addition élément par élément :")
print(f"  {a.tolist()} + {b.tolist()} = {c.tolist()}")
print()

# C'est ce qu'on fait quand on ajoute le biais !
z = torch.tensor([-1.0, 3.0])
biais = torch.tensor([0.5, -0.5])
print(f"  Ajout du biais : {z.tolist()} + {biais.tolist()} = {(z + biais).tolist()}")

In [ ]:
# ══════════════════════════════════════
# OPÉRATION 2 : Produit scalaire (dot product)
# ══════════════════════════════════════
w = torch.tensor([1.0, 0.0, -1.0])   # Poids du neurone
x = torch.tensor([2.0, 1.0, 3.0])    # Entrées

# Méthode 1 : torch.dot
dot = torch.dot(w, x)

# Méthode 2 : à la main
dot_manual = (w * x).sum()

print("Produit scalaire :")
print(f"  w = {w.tolist()}")
print(f"  x = {x.tolist()}")
print(f"  Calcul : {w[0]}×{x[0]} + {w[1]}×{x[1]} + {w[2]}×{x[2]}")
print(f"         = {w[0]*x[0]} + {w[1]*x[1]} + {w[2]*x[2]}")
print(f"         = {dot.item()}")
print(f"  torch.dot = {dot.item()}")
print(f"  (w*x).sum() = {dot_manual.item()}")

In [ ]:
# ══════════════════════════════════════
# OPÉRATION 3 : Multiplication matricielle
# ══════════════════════════════════════
W = torch.tensor([[1.0, 0.0, -1.0],    # Neurone 1
                   [2.0, -1.0, 0.0]])   # Neurone 2
x = torch.tensor([2.0, 1.0, 3.0])

# W @ x  ou  torch.matmul(W, x)  ou  W.mv(x)
result = W @ x

print("Multiplication matricielle W × x :")
print(f"  W = {W.tolist()}")
print(f"  x = {x.tolist()}")
print()
print(f"  Neurone 1 : {W[0][0]}×{x[0]} + {W[0][1]}×{x[1]} + {W[0][2]}×{x[2]}")
print(f"            = {W[0][0]*x[0]} + {W[0][1]*x[1]} + {W[0][2]*x[2]}")
print(f"            = {result[0].item()}")
print()
print(f"  Neurone 2 : {W[1][0]}×{x[0]} + {W[1][1]}×{x[1]} + {W[1][2]}×{x[2]}")
print(f"            = {W[1][0]*x[0]} + {W[1][1]*x[1]} + {W[1][2]*x[2]}")
print(f"            = {result[1].item()}")
print()
print(f"  W @ x = {result.tolist()}")
print(f"  → 2 neurones calculés en UNE opération !")

### ✏️ Exercice 1.2
Calculez le produit scalaire de `w = [2, -1, 3]` et `x = [4, 2, 1]` :
1. À la main (avec `*` et `.sum()`)
2. Avec `torch.dot()`

Résultat attendu : **9**

In [ ]:
# ══ VOTRE CODE ICI ══
w = torch.tensor([2.0, -1.0, 3.0])
x = torch.tensor([4.0, 2.0, 1.0])



---
# 🔄 Partie 2 : Le Forward Pass Complet

Le forward pass d'une couche = 3 étapes :
1. **Multiplication matricielle** : `W @ x`
2. **Ajout du biais** : `+ b`
3. **Activation** : `ReLU(z)`

Exactement comme dans l'exercice guidé du cours !

## 2.1 Forward pass à la main

In [ ]:
# ══════════════════════════════════════════════
# FORWARD PASS — Exactement comme dans la slide
# ══════════════════════════════════════════════

# Données
x = torch.tensor([2.0, 1.0, 3.0])
W = torch.tensor([[1.0, 0.0, -1.0],
                   [2.0, -1.0, 0.0]])
b = torch.tensor([0.5, -0.5])

print("╔══════════════════════════════════════╗")
print("║   FORWARD PASS — STEP BY STEP       ║")
print("╚══════════════════════════════════════╝")
print()
print(f"Entrées : x = {x.tolist()}")
print(f"Poids   : W = {W.tolist()}")
print(f"Biais   : b = {b.tolist()}")
print()

# ── Étape 1 : W × x ──
wx = W @ x
print("── Étape 1 : Produit W × x ──")
print(f"  Neurone 1 : 1×2 + 0×1 + (-1)×3 = {wx[0].item()}")
print(f"  Neurone 2 : 2×2 + (-1)×1 + 0×3 = {wx[1].item()}")
print(f"  W·x = {wx.tolist()}")
print()

# ── Étape 2 : + biais ──
z = wx + b
print("── Étape 2 : Ajout du biais ──")
print(f"  z₁ = {wx[0].item()} + {b[0].item()} = {z[0].item()}")
print(f"  z₂ = {wx[1].item()} + {b[1].item()} = {z[1].item()}")
print(f"  z = {z.tolist()}")
print()

# ── Étape 3 : ReLU ──
a = torch.relu(z)
print("── Étape 3 : Activation ReLU ──")
print(f"  a₁ = max(0, {z[0].item()}) = {a[0].item()}  {'← ÉTEINT ⛔' if a[0]==0 else '← ACTIF ✅'}")
print(f"  a₂ = max(0, {z[1].item()}) = {a[1].item()}  {'← ÉTEINT ⛔' if a[1]==0 else '← ACTIF ✅'}")
print()
print(f"  ╔═══════════════════════════╗")
print(f"  ║ SORTIE FINALE : {a.tolist()} ║")
print(f"  ╚═══════════════════════════╝")

## 2.2 Forward pass avec nn.Linear (la vraie façon PyTorch)

In [ ]:
# PyTorch encapsule W et b dans nn.Linear
torch.manual_seed(42)

couche = nn.Linear(in_features=3, out_features=2)  # 3 entrées → 2 neurones

# Remplaçons les poids par ceux de notre exercice
with torch.no_grad():
    couche.weight.copy_(torch.tensor([[1.0, 0.0, -1.0],
                                      [2.0, -1.0, 0.0]]))
    couche.bias.copy_(torch.tensor([0.5, -0.5]))

x = torch.tensor([2.0, 1.0, 3.0])

# Forward pass en UNE ligne
z = couche(x)      # = W @ x + b
a = torch.relu(z)  # activation

print("Avec nn.Linear :")
print(f"  z (avant activation) = {z.tolist()}")
print(f"  a (après ReLU)       = {a.tolist()}")
print()
print("Même résultat qu'à la main ! ✅")

### ✏️ Exercice 2.1 — Forward pass complet

Calculez le forward pass avec :
- `W = [[2, -1], [0, 4]]`
- `x = [3, 1]`
- `b = [0.5, -1]`
- Activation = ReLU

Résultat attendu : `[5.5, 3.0]`

In [ ]:
# ══ VOTRE CODE ICI ══
W = torch.tensor([[2.0, -1.0], [0.0, 4.0]])
x = torch.tensor([3.0, 1.0])
b = torch.tensor([0.5, -1.0])

# Étape 1 : W @ x

# Étape 2 : + b

# Étape 3 : ReLU


<details><summary>💡 Solution</summary>

```python
wx = W @ x         # tensor([5., 4.])
z  = wx + b        # tensor([5.5, 3.0])
a  = torch.relu(z) # tensor([5.5, 3.0]) — les deux positifs, ReLU ne change rien
print(f"Sortie : {a.tolist()}")
```
</details>

---
# ⚡ Partie 3 : Fonctions d'Activation

Sans activation, empiler des couches linéaires ne sert à rien :

$$W_2 \cdot (W_1 \cdot x + b_1) + b_2 = W' \cdot x + b'$$

L'activation **casse la linéarité** et permet au réseau d'apprendre des frontières complexes.

## 3.1 Preuve : sans activation, tout s'effondre

In [ ]:
# ══════════════════════════════════════════
# PREUVE : 2 couches linéaires = 1 couche
# ══════════════════════════════════════════

W1 = torch.tensor([[2.0, -1.0],
                    [1.0,  3.0]])
b1 = torch.tensor([0.5, -0.5])

W2 = torch.tensor([[1.0, -2.0],
                    [0.5,  1.0]])
b2 = torch.tensor([0.1, -0.3])

x = torch.tensor([1.0, 2.0])

# ── Réseau 2 couches SANS activation ──
z1 = W1 @ x + b1
z2 = W2 @ z1 + b2  # PAS d'activation entre les couches !

# ── Simplification en 1 couche ──
W_prime = W2 @ W1
b_prime = W2 @ b1 + b2
z_simple = W_prime @ x + b_prime

print("SANS activation (2 couches) :")
print(f"  z₁ = W₁·x + b₁ = {z1.tolist()}")
print(f"  z₂ = W₂·z₁ + b₂ = {z2.tolist()}")
print()
print("Simplifié en 1 couche :")
print(f"  W' = W₂·W₁ = {W_prime.tolist()}")
print(f"  b' = W₂·b₁ + b₂ = {b_prime.tolist()}")
print(f"  z  = W'·x + b' = {z_simple.tolist()}")
print()
print(f"  Résultats identiques ? {torch.allclose(z2, z_simple)}")
print("  → 2 couches sans activation = 1 couche. INUTILE ! ❌")

In [ ]:
# ── AVEC activation, c'est DIFFÉRENT ──
z1 = W1 @ x + b1
a1 = torch.relu(z1)       # ← ACTIVATION !
z2_act = W2 @ a1 + b2

print("AVEC activation (ReLU) :")
print(f"  z₁ = {z1.tolist()}")
print(f"  a₁ = ReLU(z₁) = {a1.tolist()}")
print(f"  z₂ = W₂·a₁ + b₂ = {z2_act.tolist()}")
print()
print(f"  Résultat sans activation : {z2.tolist()}")
print(f"  Résultat avec activation : {z2_act.tolist()}")
print(f"  Identiques ? {torch.allclose(z2, z2_act)}")
print("  → DIFFÉRENT ! L'activation rend la profondeur UTILE ✅")

## 3.2 Visualisation des fonctions d'activation

In [ ]:
z = torch.linspace(-5, 5, 200)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ── Sigmoid ──
axes[0].plot(z, torch.sigmoid(z), color='#7b6cf0', linewidth=2.5)
axes[0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.3)
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.3)
axes[0].set_title('Sigmoid : σ(z) = 1/(1+e⁻ᶻ)', fontweight='bold', color='#7b6cf0')
axes[0].set_ylim(-0.1, 1.1)
axes[0].fill_between(z, torch.sigmoid(z), alpha=0.1, color='#7b6cf0')

# ── ReLU ──
axes[1].plot(z, torch.relu(z), color='#2ed8a8', linewidth=2.5)
axes[1].axvline(x=0, color='gray', linestyle='--', alpha=0.3)
axes[1].set_title('ReLU : max(0, z)', fontweight='bold', color='#2ed8a8')
axes[1].fill_between(z, torch.relu(z), alpha=0.1, color='#2ed8a8')

# ── Tanh ──
axes[2].plot(z, torch.tanh(z), color='#f5c84c', linewidth=2.5)
axes[2].axhline(y=0, color='gray', linestyle='--', alpha=0.3)
axes[2].axvline(x=0, color='gray', linestyle='--', alpha=0.3)
axes[2].set_title('Tanh : (eᶻ−e⁻ᶻ)/(eᶻ+e⁻ᶻ)', fontweight='bold', color='#f5c84c')
axes[2].set_ylim(-1.2, 1.2)
axes[2].fill_between(z, torch.tanh(z), alpha=0.1, color='#f5c84c')

for ax in axes:
    ax.set_xlabel('z')
    ax.grid(alpha=0.15)

plt.tight_layout()
plt.suptitle('Les 3 fonctions d\'activation principales', fontweight='bold', y=1.02, fontsize=14)
plt.show()

In [ ]:
# ── Comparaison des valeurs pour z = -2, -1, 0, 1, 2 ──
test_z = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])

print(f"{'z':>6} | {'Sigmoid':>8} | {'ReLU':>8} | {'Tanh':>8}")
print("-" * 42)
for val in test_z:
    s = torch.sigmoid(val).item()
    r = torch.relu(val).item()
    t = torch.tanh(val).item()
    print(f"{val.item():>6.1f} | {s:>8.4f} | {r:>8.4f} | {t:>8.4f}")

---
# 📉 Partie 4 : Descente de Gradient

**L'équation :**

$$w_{new} = w - \alpha \cdot \frac{\partial L}{\partial w}$$

On minimise $L(w) = (w - 3)^2$ dont le gradient est $\frac{\partial L}{\partial w} = 2(w - 3)$

## 4.1 Descente de gradient à la main (sans PyTorch)

In [ ]:
# ══════════════════════════════════════════════
# DESCENTE DE GRADIENT — À LA MAIN
# ══════════════════════════════════════════════

def L(w):       return (w - 3) ** 2       # Fonction de perte
def dL_dw(w):   return 2 * (w - 3)        # Gradient (dérivée)

w = 0.0         # Position initiale
alpha = 0.2     # Learning rate
n_steps = 10

print("╔═══════════════════════════════════════════════════════════════════╗")
print("║  DESCENTE DE GRADIENT — CALCUL STEP BY STEP                     ║")
print("╚═══════════════════════════════════════════════════════════════════╝")
print(f"  L(w) = (w − 3)²    Minimum en w* = 3    α = {alpha}")
print()
print(f"{'Step':>4} | {'w':>10} | {'L(w)':>10} | {'∂L/∂w':>10} | {'−α×grad':>10} | {'w_new':>10}")
print("─" * 72)

history = [w]
for step in range(1, n_steps + 1):
    loss = L(w)
    grad = dL_dw(w)
    delta = -alpha * grad
    w_new = w + delta

    print(f"{step:>4} | {w:>10.4f} | {loss:>10.4f} | {grad:>10.4f} | {delta:>10.4f} | {w_new:>10.4f}")

    w = w_new
    history.append(w)

print("─" * 72)
print(f"Résultat final : w = {w:.6f}  (cible : 3.0)")
print(f"Erreur : |w - 3| = {abs(w - 3):.6f}")

In [ ]:
# ── Visualisation du trajet ──
w_range = np.linspace(-2, 8, 200)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Graphique 1 : Bille sur la courbe
ax1.plot(w_range, (w_range - 3)**2, color='#7b6cf0', linewidth=2, label='L(w) = (w−3)²')
ax1.fill_between(w_range, (w_range - 3)**2, alpha=0.08, color='#7b6cf0')
for i, w in enumerate(history):
    alpha_dot = 0.3 + 0.7 * (i / len(history))
    ax1.plot(w, L(w), 'o', color='#34d8d8', markersize=8, alpha=alpha_dot)
# Trajet
for i in range(len(history)-1):
    ax1.annotate('', xy=(history[i+1], L(history[i+1])),
                 xytext=(history[i], L(history[i])),
                 arrowprops=dict(arrowstyle='->', color='#34d8d8', alpha=0.5))
ax1.axvline(x=3, color='#2ed8a8', linestyle='--', alpha=0.3, label='w* = 3')
ax1.set_xlabel('w'); ax1.set_ylabel('L(w)')
ax1.set_title('Trajet de la descente de gradient', fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.15)

# Graphique 2 : Convergence de la perte
losses = [L(w) for w in history]
ax2.plot(range(len(losses)), losses, 'o-', color='#2ed8a8', linewidth=2, markersize=6)
ax2.fill_between(range(len(losses)), losses, alpha=0.1, color='#2ed8a8')
ax2.set_xlabel('Step'); ax2.set_ylabel('L(w)')
ax2.set_title('Convergence de la perte', fontweight='bold')
ax2.grid(alpha=0.15)

plt.tight_layout()
plt.show()

## 4.2 Descente de gradient avec autograd PyTorch

In [ ]:
# ══════════════════════════════════════════
# AUTOGRAD — PyTorch calcule le gradient automatiquement !
# ══════════════════════════════════════════

# requires_grad=True dit à PyTorch de suivre les opérations
w = torch.tensor(0.0, requires_grad=True)
alpha = 0.2

print("Avec autograd (PyTorch calcule ∂L/∂w tout seul) :")
print()

for step in range(1, 11):
    # Forward : calculer la perte
    loss = (w - 3) ** 2

    # Backward : calculer automatiquement ∂L/∂w
    loss.backward()

    # Le gradient est stocké dans w.grad
    grad = w.grad.item()

    # Mise à jour (dans un bloc no_grad car ce n'est pas un calcul à tracer)
    with torch.no_grad():
        w -= alpha * w.grad

    # IMPORTANT : remettre le gradient à zéro
    w.grad.zero_()

    print(f"  Step {step:2d}: w = {w.item():8.4f}  L = {loss.item():8.4f}  grad = {grad:8.4f}")

print(f"\nRésultat : w = {w.item():.6f}")
print("Identique au calcul à la main ! ✅")

---
# 🎯 Partie 5 : Impact du Learning Rate

Le learning rate α contrôle la taille du pas :
- **Trop petit** → convergence très lente
- **Optimal** → convergence rapide et stable
- **Trop grand** → oscillations ou DIVERGENCE

In [ ]:
# ══════════════════════════════════════════
# COMPARAISON : 3 learning rates
# ══════════════════════════════════════════

learning_rates = [0.02, 0.20, 0.90]
colors = ['#4aa4ff', '#2ed8a8', '#f25e5e']
labels = ['α=0.02 (trop petit)', 'α=0.20 (optimal)', 'α=0.90 (trop grand)']
n_steps = 30

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

w_range = np.linspace(-2, 8, 200)
ax1.plot(w_range, (w_range-3)**2, color='#7b6cf0', linewidth=2, alpha=0.5)

for lr, col, lab in zip(learning_rates, colors, labels):
    w = 0.0
    losses = [L(w)]
    ws = [w]

    for _ in range(n_steps):
        w = w - lr * dL_dw(w)
        ws.append(w)
        losses.append(L(w))

    # Trajet sur la courbe
    ax1.plot([w for w in ws[:15]], [L(w) for w in ws[:15]], 'o-',
             color=col, markersize=4, linewidth=1.5, label=lab, alpha=0.8)

    # Convergence
    ax2.plot(range(len(losses)), losses, '-', color=col, linewidth=2, label=lab)

ax1.axvline(x=3, color='#2ed8a8', linestyle='--', alpha=0.2)
ax1.set_xlabel('w'); ax1.set_ylabel('L(w)'); ax1.set_ylim(-1, 35)
ax1.set_title('Trajet sur L(w)', fontweight='bold')
ax1.legend(fontsize=9); ax1.grid(alpha=0.15)

ax2.set_xlabel('Step'); ax2.set_ylabel('L(w)'); ax2.set_ylim(-0.5, 15)
ax2.set_title('Convergence de la perte', fontweight='bold')
ax2.legend(fontsize=9); ax2.grid(alpha=0.15)

plt.tight_layout()
plt.show()

print("Observations :")
print("  🔵 α=0.02 : rampe lentement, toujours loin du minimum après 30 steps")
print("  🟢 α=0.20 : converge vite et proprement")
print("  🔴 α=0.90 : OSCILLE ! Saute d'un côté à l'autre du minimum")

## 5.1 💥 Divergence : quand α > 1.0

In [ ]:
# ══════════════════════════════════════════
# 💥 DIVERGENCE
# ══════════════════════════════════════════

print("💥 DIVERGENCE avec α = 1.05")
print("=" * 50)
print()

w = 0.0
alpha_div = 1.05

losses_div = []
for step in range(1, 11):
    loss = L(w)
    grad = dL_dw(w)
    w_new = w - alpha_div * grad

    status = "📈 AUGMENTE !" if step > 1 and L(w_new) > loss else "📉 diminue"

    print(f"Step {step}: w = {w:>10.2f} → grad = {grad:>10.2f} → w_new = {w_new:>10.2f}  L = {L(w_new):>12.1f}  {status}")

    losses_div.append(L(w_new))
    w = w_new

print()
print(f"Après 10 steps : w = {w:.0f}  (au lieu de 3 !)")
print(f"La perte est passée de 9.0 à {losses_div[-1]:.0f} — elle a EXPLOSÉ ! 💥")

In [ ]:
# ── Visualisation de la divergence ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Barres de perte
steps = range(1, len(losses_div)+1)
colors_bar = ['#f25e5e' if i > 0 and losses_div[i] > losses_div[i-1] else '#2ed8a8'
              for i in range(len(losses_div))]
ax1.bar(steps, losses_div[:10], color=colors_bar[:10], alpha=0.7)
ax1.set_xlabel('Step'); ax1.set_ylabel('L(w)')
ax1.set_title('💥 La perte EXPLOSE (α=1.05)', fontweight='bold', color='#f25e5e')
ax1.grid(alpha=0.15)

# Comparaison converge vs diverge
w_c, w_d = 0.0, 0.0
lc, ld = [L(0)], [L(0)]
for _ in range(20):
    w_c = w_c - 0.2 * dL_dw(w_c)
    w_d = w_d - 1.05 * dL_dw(w_d)
    lc.append(L(w_c))
    ld.append(min(L(w_d), 1000))  # Cap pour la visu

ax2.plot(lc, color='#2ed8a8', linewidth=2.5, label='α=0.20 (converge)')
ax2.plot(ld, color='#f25e5e', linewidth=2.5, label='α=1.05 (diverge !)')
ax2.set_xlabel('Step'); ax2.set_ylabel('L(w)'); ax2.set_ylim(-5, 100)
ax2.set_title('Convergence vs Divergence', fontweight='bold')
ax2.legend(); ax2.grid(alpha=0.15)

plt.tight_layout()
plt.show()

### ✏️ Exercice 5.1
Testez α = 0.5 sur L(w) = (w-3)². Combien de steps faut-il pour atteindre exactement w = 3 ?

In [ ]:
# ══ VOTRE CODE ICI ══
w = 0.0
alpha = 0.5



<details><summary>💡 Solution</summary>

```python
w = 0.0
for step in range(1, 6):
    w = w - 0.5 * 2 * (w - 3)
    print(f"Step {step}: w = {w}")
# Step 1: w = 3.0 → UN SEUL STEP !
# α = 0.5 = 1/(2*1) = 1/L'' est le learning rate OPTIMAL pour cette fonction.
```
</details>

---
# 🔀 Partie 6 : Le Problème XOR

XOR = "ou exclusif" : vrai quand les entrées sont **différentes**.

| x₁ | x₂ | XOR |
|----|----|-----|
| 0  | 0  | 0   |
| 0  | 1  | 1   |
| 1  | 0  | 1   |
| 1  | 1  | 0   |

**Impossible** à résoudre avec un seul neurone (pas de droite qui sépare les classes).

## 6.1 Visualisation du problème

In [ ]:
# ── Les 4 points XOR ──
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
Y = torch.tensor([[0.], [1.], [1.], [0.]])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Points
for i in range(4):
    color = '#2ed8a8' if Y[i] == 1 else '#f25e5e'
    marker = 'o'
    ax1.scatter(X[i, 0], X[i, 1], c=color, s=200, zorder=5, edgecolors='white', linewidth=2)
    ax1.annotate(f'  XOR={int(Y[i].item())}', (X[i, 0], X[i, 1]), fontsize=11, fontweight='bold', color=color)
    ax2.scatter(X[i, 0], X[i, 1], c=color, s=200, zorder=5, edgecolors='white', linewidth=2)

# Droite qui essaie de séparer (ax1)
x_line = np.linspace(-0.5, 1.5, 100)
ax1.plot(x_line, 0.6 - x_line, '--', color='#f25e5e', alpha=0.5, linewidth=2, label='Droite impossible')
ax1.set_title('❌ 1 neurone = droite → ÉCHOUE', fontweight='bold', color='#f25e5e')
ax1.legend(fontsize=9)

ax2.set_title('✅ 2 neurones + ReLU → RÉSOUT', fontweight='bold', color='#2ed8a8')

for ax in [ax1, ax2]:
    ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
    ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
    ax.grid(alpha=0.15); ax.set_aspect('equal')

plt.tight_layout()
plt.show()

## 6.2 XOR à la main : calcul step by step

In [ ]:
# ══════════════════════════════════════════════
# RÉSOLUTION XOR — CALCUL COMPLET
# ══════════════════════════════════════════════

# Poids de la couche cachée (2 neurones)
W1 = torch.tensor([[1., 1.],     # h₁ : détecte "au moins un 1"
                    [-1., -1.]])  # h₂ : détecte "pas les deux à 1"
b1 = torch.tensor([-0.5, 1.5])

# Poids de la couche de sortie
W2 = torch.tensor([[1., -1.]])
b2 = torch.tensor([0.5])

print("╔══════════════════════════════════════════╗")
print("║  RÉSOLUTION XOR — 4 CAS                 ║")
print("╚══════════════════════════════════════════╝")
print()

for i in range(4):
    x = X[i]
    expected = int(Y[i].item())

    # Couche cachée
    z1 = W1 @ x + b1
    h = torch.relu(z1)

    # Sortie
    z2 = W2 @ h + b2
    prediction = 1 if z2.item() > 0.3 else 0

    correct = "✅" if prediction == expected else "❌"

    print(f"── XOR({int(x[0])}, {int(x[1])}) ──  Attendu = {expected}")
    print(f"   z₁ = W₁·x + b₁ = {z1.tolist()}")
    print(f"   h  = ReLU(z₁)   = {h.tolist()}")
    print(f"   y  = W₂·h + b₂  = {z2.item():.2f}  → prédit: {prediction}  {correct}")
    print()

## 6.3 🚀 Entraîner un réseau XOR avec PyTorch (la vraie méthode)

In [ ]:
# ══════════════════════════════════════════════
# RÉSEAU DE NEURONES COMPLET POUR XOR
# ══════════════════════════════════════════════

torch.manual_seed(42)

# Données XOR
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
Y = torch.tensor([[0.], [1.], [1.], [0.]])

# Définition du réseau
model = nn.Sequential(
    nn.Linear(2, 4),     # Couche cachée : 2 entrées → 4 neurones
    nn.ReLU(),           # Activation
    nn.Linear(4, 1),     # Sortie : 4 → 1
    nn.Sigmoid()         # Sigmoid pour avoir une probabilité [0, 1]
)

print("Architecture du réseau :")
print(model)
print()

# Nombre de paramètres
n_params = sum(p.numel() for p in model.parameters())
print(f"Nombre de paramètres : {n_params}")
for name, param in model.named_parameters():
    print(f"  {name:>10} : shape {list(param.shape)} = {param.numel()} valeurs")

In [ ]:
# ══════════════════════════════════════════════
# ENTRAÎNEMENT
# ══════════════════════════════════════════════

criterion = nn.BCELoss()                          # Binary Cross Entropy
optimizer = optim.Adam(model.parameters(), lr=0.05)  # Adam optimizer

losses = []
n_epochs = 2000

for epoch in range(n_epochs):
    # Forward pass
    pred = model(X)
    loss = criterion(pred, Y)

    # Backward pass
    optimizer.zero_grad()   # Remettre les gradients à zéro
    loss.backward()         # Calculer les gradients
    optimizer.step()        # Mettre à jour les poids

    losses.append(loss.item())

    if epoch % 400 == 0 or epoch == n_epochs - 1:
        preds = (pred > 0.5).float()
        acc = (preds == Y).float().mean()
        print(f"Epoch {epoch:>4d} | Loss = {loss.item():.4f} | Accuracy = {acc.item()*100:.0f}%")

print("\n✅ Entraînement terminé !")

In [ ]:
# ── Courbe de perte ──
plt.figure(figsize=(10, 4))
plt.plot(losses, color='#7b6cf0', linewidth=1.5, alpha=0.7)
plt.fill_between(range(len(losses)), losses, alpha=0.1, color='#7b6cf0')
plt.xlabel('Epoch'); plt.ylabel('Loss (BCE)')
plt.title('Courbe d\'apprentissage — XOR', fontweight='bold')
plt.grid(alpha=0.15)
plt.show()

In [ ]:
# ── Résultats finaux ──
print("Résultats du réseau entraîné :")
print()
with torch.no_grad():
    preds = model(X)
    for i in range(4):
        x1, x2 = int(X[i, 0]), int(X[i, 1])
        expected = int(Y[i].item())
        output = preds[i].item()
        predicted = 1 if output > 0.5 else 0
        ok = "✅" if predicted == expected else "❌"
        print(f"  XOR({x1}, {x2}) = {output:.4f}  →  {predicted}  (attendu: {expected})  {ok}")

In [ ]:
# ══════════════════════════════════════════════
# VISUALISATION : Frontière de décision apprise
# ══════════════════════════════════════════════

# Grille de points
xx = np.linspace(-0.5, 1.5, 200)
yy = np.linspace(-0.5, 1.5, 200)
XX, YY = np.meshgrid(xx, yy)
grid = torch.tensor(np.c_[XX.ravel(), YY.ravel()], dtype=torch.float32)

with torch.no_grad():
    ZZ = model(grid).reshape(200, 200).numpy()

plt.figure(figsize=(7, 6))
plt.contourf(XX, YY, ZZ, levels=50, cmap='RdYlGn', alpha=0.6)
plt.contour(XX, YY, ZZ, levels=[0.5], colors='white', linewidths=2)
plt.colorbar(label='Sortie du réseau')

# Points XOR
for i in range(4):
    color = '#2ed8a8' if Y[i] == 1 else '#f25e5e'
    plt.scatter(X[i, 0], X[i, 1], c=color, s=250, zorder=5,
               edgecolors='white', linewidth=2.5)
    plt.annotate(f' XOR={int(Y[i].item())}', (X[i, 0]+0.05, X[i, 1]+0.05),
                fontsize=11, fontweight='bold', color='white')

plt.xlabel('x₁', fontsize=12); plt.ylabel('x₂', fontsize=12)
plt.title('Frontière de décision apprise par le réseau', fontweight='bold', fontsize=14)
plt.grid(alpha=0.1)
plt.tight_layout()
plt.show()

print("La ligne blanche = frontière de décision (sortie = 0.5)")
print("Vert = classe 1 (XOR=1), Rouge = classe 0 (XOR=0)")
print("→ Le réseau a appris une frontière NON-LINÉAIRE ! ✅")

## 6.4 Bonus : Comparaison SGD vs Adam

In [ ]:
# ══════════════════════════════════════════════
# SGD vs ADAM — Qui converge plus vite ?
# ══════════════════════════════════════════════

def train_xor(optimizer_class, lr, epochs=2000):
    torch.manual_seed(42)
    model = nn.Sequential(nn.Linear(2, 4), nn.ReLU(), nn.Linear(4, 1), nn.Sigmoid())
    criterion = nn.BCELoss()
    optimizer = optimizer_class(model.parameters(), lr=lr)
    losses = []
    for _ in range(epochs):
        pred = model(X)
        loss = criterion(pred, Y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses

losses_sgd  = train_xor(optim.SGD, lr=0.5)
losses_adam = train_xor(optim.Adam, lr=0.05)

plt.figure(figsize=(10, 4))
plt.plot(losses_sgd, color='#f5c84c', linewidth=1.5, alpha=0.7, label='SGD (lr=0.5)')
plt.plot(losses_adam, color='#2ed8a8', linewidth=1.5, alpha=0.7, label='Adam (lr=0.05)')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('SGD vs Adam sur XOR', fontweight='bold')
plt.legend(); plt.grid(alpha=0.15)
plt.show()

print(f"Loss finale SGD  : {losses_sgd[-1]:.4f}")
print(f"Loss finale Adam : {losses_adam[-1]:.4f}")
print("\nAdam adapte le learning rate par paramètre → converge plus vite !")

---
# 🏆 Exercice Final : Réseau Complet

Entraînez un réseau pour apprendre la fonction **cercle** :
- Classe 1 si $x_1^2 + x_2^2 < 0.5$
- Classe 0 sinon

C'est un problème non-linéaire (comme XOR mais en continu).

In [ ]:
# ── Données ──
torch.manual_seed(42)
N = 500
X_circle = torch.randn(N, 2)
Y_circle = ((X_circle[:, 0]**2 + X_circle[:, 1]**2) < 1.0).float().unsqueeze(1)

plt.figure(figsize=(6, 6))
plt.scatter(X_circle[Y_circle.squeeze()==1, 0], X_circle[Y_circle.squeeze()==1, 1],
           c='#2ed8a8', s=15, alpha=0.6, label='Classe 1 (intérieur)')
plt.scatter(X_circle[Y_circle.squeeze()==0, 0], X_circle[Y_circle.squeeze()==0, 1],
           c='#f25e5e', s=15, alpha=0.6, label='Classe 0 (extérieur)')
plt.title('Données : Classification Cercle', fontweight='bold')
plt.legend(); plt.grid(alpha=0.15); plt.axis('equal')
plt.show()

In [ ]:
# ══ VOTRE CODE ICI ══
#
# 1. Créez un réseau avec :
#    - Couche cachée : 2 → 16 neurones + ReLU
#    - Sortie : 16 → 1 + Sigmoid
#
# 2. Entraînez-le avec Adam (lr=0.01) pendant 1000 epochs
#
# 3. Visualisez la frontière de décision
#
# Indices :
#   model = nn.Sequential(...)
#   criterion = nn.BCELoss()
#   optimizer = optim.Adam(model.parameters(), lr=0.01)



<details><summary>💡 Solution complète</summary>

```python
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(2, 16), nn.ReLU(),
    nn.Linear(16, 1), nn.Sigmoid()
)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

for epoch in range(1000):
    pred = model(X_circle)
    loss = criterion(pred, Y_circle)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 200 == 0:
        acc = ((pred > 0.5).float() == Y_circle).float().mean()
        print(f"Epoch {epoch}: Loss={loss.item():.4f} Acc={acc.item()*100:.0f}%")

# Visualisation
xx = np.linspace(-3, 3, 200)
XX, YY = np.meshgrid(xx, xx)
grid = torch.tensor(np.c_[XX.ravel(), YY.ravel()], dtype=torch.float32)
with torch.no_grad():
    ZZ = model(grid).reshape(200, 200).numpy()
plt.contourf(XX, YY, ZZ, levels=50, cmap='RdYlGn', alpha=0.5)
plt.contour(XX, YY, ZZ, levels=[0.5], colors='white', linewidths=2)
plt.scatter(X_circle[Y_circle.squeeze()==1, 0], X_circle[Y_circle.squeeze()==1, 1], c='#2ed8a8', s=10)
plt.scatter(X_circle[Y_circle.squeeze()==0, 0], X_circle[Y_circle.squeeze()==0, 1], c='#f25e5e', s=10)
plt.title('Frontière apprise : Cercle'); plt.axis('equal'); plt.show()
```
</details>

---
# 📝 Résumé

| Concept | Ce qu'on a appris | Code PyTorch |
|---------|------------------|--------------|
| **Tenseurs** | Conteneurs de nombres avec dimensions | `torch.tensor()`, `.shape`, `.dim()` |
| **Produit scalaire** | Σ wᵢxᵢ = un nombre | `torch.dot(w, x)` |
| **Matmul** | Toute une couche en 1 opération | `W @ x` ou `torch.matmul()` |
| **Forward pass** | W·x + b → activation | `nn.Linear()` + `torch.relu()` |
| **Activation** | Casse la linéarité | `nn.ReLU()`, `nn.Sigmoid()`, `nn.Tanh()` |
| **Gradient** | Direction de la plus forte pente | `loss.backward()` → `w.grad` |
| **Descente** | w = w − α × ∂L/∂w | `optimizer.step()` |
| **Learning rate** | Trop petit=lent, trop grand=diverge | `optim.Adam(lr=...)` |
| **XOR** | 1 neurone échoue, 2+ReLU résolvent | `nn.Sequential(...)` |

